[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session6/solutions/Lab3_Agentic_RAG_LangGraph_Solutions.ipynb)


# Lab 3 SOLUTIONS: Agentic RAG with LangGraph
## CCI Session 6

**Duration:** 15 minutes

### Clinical scenario
> Basic RAG retrieves once and answers. Complex clinical questions benefit from **query rewriting**, **retrieval grading**, and **controlled retries** before generation.

### Objectives
- Build a **LangGraph** agentic RAG workflow
- Add query rewriting and document grading (self-RAG style)
- Compare answers and metrics against **basic RAG** from Lab 1

### Design choices (agentic RAG)
- **Query rewrite**: maps vague user language to retrieval-friendly clinical terms; can retry with a different angle.
- **Grade documents**: cheap self-check — are these chunks actually about the question? If not, rewrite and retrieve again.
- **Max iterations**: caps cost/latency; then we still **generate** with best-effort context.

**Chunking / embeddings / vector DB** use Lab 1 defaults here so you mostly isolate the effect of the **agent workflow**.


---
## Setup

Install packages, load **Colab secrets** (`OPENAI_API_KEY`, `LLAMA_CLOUD_API_KEY`), upload **WT.pdf**, then continue. Restart runtime if needed after `pip install`.


In [ ]:
!pip install -q llama-parse langchain langchain-openai langchain-community langchain-chroma chromadb langgraph deepeval

In [ ]:
import os
from google.colab import userdata, files
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['LLAMA_CLOUD_API_KEY'] = userdata.get('LLAMA_CLOUD_API_KEY')
files.upload()
PDF_PATH = '/content/WT.pdf'

---
## Cell 1 — Retriever (same stack as Lab 1)

**Chroma** + **`text-embedding-3-small`** + **recursive chunking 1000/200** + **`k=4`**. The agent loop will rewrite queries and optionally re-retrieve; the vector layer stays a standard baseline.


In [ ]:
from llama_parse import LlamaParse
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

raw = LlamaParse(api_key=os.environ['LLAMA_CLOUD_API_KEY'], result_type='markdown').load_data(PDF_PATH)
lc_docs = [Document(page_content=d.text, metadata={'source': 'WT.pdf'}) for d in raw]
chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(lc_docs)
vs = Chroma.from_documents(chunks, OpenAIEmbeddings(model='text-embedding-3-small'), collection_name='wt_agentic')
retriever = vs.as_retriever(search_kwargs={'k': 4})
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

## Cell 2 — State

In [ ]:
from typing import TypedDict, List
class AgenticRAGState(TypedDict):
    question: str
    rewritten_question: str
    documents: List[str]
    grade: str
    answer: str
    iteration: int

## Cell 3 — Nodes

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

rewrite_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Rewrite the user question to be precise, clinically-specific, and unambiguous for retrieval against a Wilms tumor treatment guideline. Return only the rewritten question.'),
    ('human', 'Original: {q}\nIteration: {i}\nRewrite (try a different angle if iteration > 1):')
])

def rewrite_query_node(state):
    i = state.get('iteration', 0) + 1
    out = (rewrite_prompt | llm).invoke({'q': state['question'], 'i': i})
    return {'rewritten_question': out.content.strip(), 'iteration': i}

def retrieve_node(state):
    docs = retriever.invoke(state['rewritten_question'])
    return {'documents': [d.page_content for d in docs]}

grade_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You judge if retrieved context is RELEVANT to the question. Reply with exactly one word: relevant OR irrelevant.'),
    ('human', 'Question: {q}\n\nContext:\n{ctx}')
])

def grade_documents_node(state):
    ctx = '\n\n'.join(state['documents'])[:4000]
    out = (grade_prompt | llm).invoke({'q': state['question'], 'ctx': ctx})
    g = 'relevant' if 'relevant' in out.content.lower() and 'irrelevant' not in out.content.lower() else 'irrelevant'
    return {'grade': g}

answer_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a pediatric oncology clinical assistant. Answer using ONLY the provided context. If the answer is not present, say so.\n\nContext:\n{ctx}'),
    ('human', '{q}')
])

def generate_answer_node(state):
    ctx = '\n\n'.join(state['documents'])
    out = (answer_prompt | llm).invoke({'ctx': ctx, 'q': state['question']})
    return {'answer': out.content}

## Cell 4 — Edges

In [ ]:
MAX_ITER = 2
def route_after_grade(state):
    if state['grade'] == 'relevant':
        return 'generate'
    if state.get('iteration', 0) < MAX_ITER:
        return 'rewrite'
    return 'generate'  # best effort

## Cell 5 — Compile graph

In [ ]:
from langgraph.graph import StateGraph, START, END

g = StateGraph(AgenticRAGState)
g.add_node('rewrite', rewrite_query_node)
g.add_node('retrieve', retrieve_node)
g.add_node('grade', grade_documents_node)
g.add_node('generate', generate_answer_node)
g.add_edge(START, 'rewrite')
g.add_edge('rewrite', 'retrieve')
g.add_edge('retrieve', 'grade')
g.add_conditional_edges('grade', route_after_grade, {'generate': 'generate', 'rewrite': 'rewrite'})
g.add_edge('generate', END)
agentic_app = g.compile()

## Cell 6 — Test

In [ ]:
questions = [
    'Show me Wilms tumor staging that requires both chemo and radiation',
    'What are the late effects of treatment in Stage I patients?',
    'Compare treatment regimens for favorable vs unfavorable histology',
]
for q in questions:
    out = agentic_app.invoke({'question': q, 'iteration': 0})
    print('Q:', q)
    print('A:', out['answer'])
    print('iterations:', out['iteration'])
    print('-' * 60)

## Cell 7 — Compare with basic RAG

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

basic_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Answer ONLY from the context.\n\n{context}'),
    ('human', '{input}')
])
basic_chain = create_retrieval_chain(retriever, create_stuff_documents_chain(llm, basic_prompt))

for q in questions:
    basic = basic_chain.invoke({'input': q})['answer']
    agentic = agentic_app.invoke({'question': q, 'iteration': 0})['answer']
    print(f'\nQ: {q}\n--BASIC--\n{basic}\n--AGENTIC--\n{agentic}\n')

## Cell 8 — DeepEval comparison

In [ ]:
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
import pandas as pd

metrics = [FaithfulnessMetric(threshold=0.7, model='gpt-4o-mini'),
           AnswerRelevancyMetric(threshold=0.7, model='gpt-4o-mini')]

rows = []
for q in questions:
    b = basic_chain.invoke({'input': q})
    a = agentic_app.invoke({'question': q, 'iteration': 0})
    cb = LLMTestCase(input=q, actual_output=b['answer'], retrieval_context=[d.page_content for d in b['context']])
    ca = LLMTestCase(input=q, actual_output=a['answer'], retrieval_context=a['documents'])
    row = {'question': q[:50]}
    for m in metrics:
        m.measure(cb); row[f'basic_{m.__class__.__name__}'] = round(m.score, 3)
        m.measure(ca); row[f'agentic_{m.__class__.__name__}'] = round(m.score, 3)
    rows.append(row)
print(pd.DataFrame(rows))